In [0]:
!pip install -r requirements.txt
%restart_python

In [0]:
import pandas as pd

csv_path = "/Volumes/abs_data/default/raw_volume/Emails.csv"
df = pd.read_csv(csv_path)

f_df = df[df["ExtractedBodyText"].str.len() >= 800]

def get_value(row_index):
    return f_df.iloc[row_index]["ExtractedBodyText"]



In [0]:
from shared.llm_utils import llm_basic, llm_structured, format_system_prompt
from pydantic import BaseModel

class EmailSummary(BaseModel):
    summary: str
    search_phrases: list[str]
    persons_of_interests: list[str]

print(format_system_prompt("You are provided an email to analyse and your task is to extract the following details: detailed summary, search phrases (ie. 'geopolitics in the middle east', 'government inefficiencies', 'climate change and the environment') which can be used to categorise the email. I also add some general rephrasing versions of each of the search phrases. output wiill just include {summary: str, search_phrases: list[str],  persons_of_interests: list[str]}", model="gpt-5-mini"))

In [0]:
ROW_NUMBER = 55

value = get_value(ROW_NUMBER)
print(value)

In [0]:
EMAIL_PROMPT = """ \
<instructions>
You are an expert email analyst and information extractor. Your job is to read a single input email and produce a compact, precise JSON object with three fields: {summary: str, search_phrases: list[str], persons_of_interests: list[str]}.

Follow these rules exactly:
- Output only valid JSON (no surrounding commentary, no markdown, no extra keys).
- Do not hallucinate. Only include information directly supported by the email text.
- Reason internally step-by-step if needed, but do NOT output your chain-of-thought—only the final JSON.
- Use neutral, professional language in the summary.
</instructions>

<input_format>
You will be given the full email text. The email may include headers, signatures, dates, or inline content. Use all available content to extract facts.
</input_format>

<output_requirements>
1) summary (string)
- Provide a focused, detailed summary of the email: sender intent, main points, requests or actions requested, relevant facts (dates, locations, organizations, metrics, deadlines), and tone (e.g., urgent, informational, collaborative).
- Keep the summary concise but detailed (approximately 50–150 words). Use complete sentences.

2) search_phrases (array of strings)
- Produce 6–12 short topical search phrases that capture the email's themes. Each phrase should be 2–6 words, lowercase, no punctuation at ends.
- For each distinct theme, include 1–3 paraphrased variants (e.g., "climate policy", "environmental regulation", "climate change policy").
- Prefer broad-to-specific phrasing that works for categorization and search indexing (subjects, issues, sectors, actions).
- Do not include full sentences—only short keyword-style phrases.

3) persons_of_interests (array of strings)
- List named individuals mentioned in the email who are relevant for follow-up or categorization.
- Use the name as it appears in the email; you may append a short parenthetical role or organization only if it is explicitly stated in the email (e.g., "Jane Doe (Director, X)").
- If none, return an empty list [].

General constraints:
- Use American English spelling.
- Do not include any explanation or additional fields.
- If a required field has no content, set it to an empty string ("") for summary or an empty list [] for the arrays.
</output_requirements>
    """

summarise = llm_structured(user_prompt=value, system_prompt=EMAIL_PROMPT, text_format=EmailSummary)
display(summarise)